# Denoising autoencoder con masking por bloque, DS2OS

Capa propia que anula de una vez todas las columnas one-hot de una misma variable categórica, activa solo en entrenamiento. Cada variable numérica forma su propio bloque. El nivel 0 es el control sin corrupción.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path.home() / ".cache/kagglehub/datasets/libamariyam/ds2os-dataset/versions/1/DS2OS.csv"

df = pd.read_csv(DATA_PATH).drop(columns=['timestamp'])
df['accessedNodeType'] = df['accessedNodeType'].fillna('missing')
df['value'] = df['value'].replace({'true': 1, 'false': 0, 'twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
print(df.shape)

(357952, 12)


In [3]:
CAT_COLS = ['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
            'destinationServiceAddress', 'destinationServiceType', 'destinationLocation',
            'accessedNodeAddress', 'accessedNodeType', 'operation']

y = (df['normality'] != 'normal').astype(int).values
X = pd.get_dummies(df.drop(columns=['normality']), columns=CAT_COLS)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train_arr = scaler.transform(X_train)
X_test_arr = scaler.transform(X_test)
X_train_normal = X_train_arr[y_train == 0]

print(f"X_train: {X_train_arr.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test_arr.shape}")

X_train: (286361, 505), X_train_normal: (278347, 505), X_test: (71591, 505)


In [4]:
# Cada columna one-hot pertenece al bloque de su variable original y 'value' forma su propio bloque
columns = list(X.columns)
group_of_column = []
for col in columns:
    matched = None
    for cat in CAT_COLS:
        if col.startswith(cat + '_'):
            matched = cat
            break
    group_of_column.append(matched if matched is not None else col)

groups = sorted(set(group_of_column), key=lambda g: group_of_column.index(g))
group_index = {g: i for i, g in enumerate(groups)}
num_groups = len(groups)

group_matrix = np.zeros((num_groups, len(columns)), dtype=np.float32)
for col_idx, g in enumerate(group_of_column):
    group_matrix[group_index[g], col_idx] = 1.0

print(f"Columnas: {len(columns)}, bloques: {num_groups}")
print(f"Bloques: {groups}")
print(f"Tamaño de cada bloque: {group_matrix.sum(axis=1).astype(int).tolist()}")

Columnas: 505, bloques: 11
Bloques: ['value', 'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation', 'destinationServiceAddress', 'destinationServiceType', 'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation']
Tamaño de cada bloque: [1, 84, 89, 8, 21, 85, 8, 21, 170, 13, 5]


In [5]:
class BlockMasking(layers.Layer):
    """Anula con probabilidad mask_rate bloques completos de columnas.

    group_matrix (num_groups, input_dim) indica a qué bloque pertenece cada columna.
    """
    def __init__(self, mask_rate, group_matrix, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate
        self.group_matrix_np = group_matrix
        self.group_matrix = tf.constant(group_matrix, dtype=tf.float32)
        self.num_groups = group_matrix.shape[0]

    def call(self, inputs, training=None):
        if training:
            batch_size = tf.shape(inputs)[0]
            group_keep = tf.cast(
                tf.random.uniform((batch_size, self.num_groups)) >= self.mask_rate,
                tf.float32,
            )
            col_keep = tf.matmul(group_keep, self.group_matrix)
            return inputs * col_keep
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({'mask_rate': self.mask_rate})
        return config


def build_dae_block_masking(input_dim, hidden_units, mask_rate, group_matrix):
    inputs = keras.Input(shape=(input_dim,))
    x = BlockMasking(mask_rate, group_matrix)(inputs)
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)


HIDDEN_UNITS = [128, 64, 16]
MASK_RATES = [0.0, 0.05, 0.1, 0.2, 0.3]

results = {}
for rate in MASK_RATES:
    tf.keras.utils.set_random_seed(SEED)
    model = build_dae_block_masking(X_train_arr.shape[1], HIDDEN_UNITS, rate, group_matrix)
    history = model.fit(
        X_train_normal, X_train_normal,
        epochs=100,
        batch_size=256,
        validation_split=0.1,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
        verbose=0,
    )
    auc = roc_auc_score(y_test, reconstruction_error(model, X_test_arr))
    results[rate] = {'model': model, 'auc': auc, 'epochs': len(history.history['loss'])}
    print(f"mask_rate={rate}: epocas={results[rate]['epochs']}, ROC-AUC={auc:.4f}")

mask_rate=0.0: epocas=99, ROC-AUC=0.9211


mask_rate=0.05: epocas=100, ROC-AUC=0.9392


mask_rate=0.1: epocas=100, ROC-AUC=0.8933


mask_rate=0.2: epocas=100, ROC-AUC=0.9548


mask_rate=0.3: epocas=100, ROC-AUC=0.8033


In [6]:
best_rate = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_rate]['model']
print(f"Mejor variante (block masking): mask_rate={best_rate} (ROC-AUC={results[best_rate]['auc']:.4f})")
print(f"Control (mask_rate=0.0): ROC-AUC={results[0.0]['auc']:.4f}")
print(f"Diferencia mejor block-masking vs control: {results[best_rate]['auc'] - results[0.0]['auc']:+.4f}")

threshold = np.percentile(reconstruction_error(best_model, X_train_normal), 95)
y_pred = (reconstruction_error(best_model, X_test_arr) > threshold).astype(int)

print()
print(classification_report(y_test, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC (mejor variante block masking): {results[best_rate]['auc']:.4f}")

Mejor variante (block masking): mask_rate=0.2 (ROC-AUC=0.9548)
Control (mask_rate=0.0): ROC-AUC=0.9211
Diferencia mejor block-masking vs control: +0.0337



              precision    recall  f1-score   support

      normal       0.99      0.97      0.98     69588
      ataque       0.37      0.70      0.48      2003

    accuracy                           0.96     71591
   macro avg       0.68      0.83      0.73     71591
weighted avg       0.97      0.96      0.96     71591

[[67166  2422]
 [  601  1402]]
ROC-AUC (mejor variante block masking): 0.9548
